# Programación declarada por ruta y fecha

Una fila representa una ruta en una fecha de servicio.
Los viajes sin frecuencias y las plantillas por frecuencia permanecen separados.
Los valores iniciales corresponden a la primera entrega 2026 y son parámetros de ejecución.
Consulta docs/contracts/route_service_day.md y docs/gold_route_service_day_execution.md.
Ejecuta **Run all** en modo validate antes de publicar.


In [ ]:
# BLOQUE 1 — Repositorio, identidad y horizonte de programación
from pathlib import Path
import sys
import json

defaults = {
    "repo_root": "", "catalog": "workspace", "schema": "gtfs_bogota",
    "feed_version": "2026-09-29",
    "snapshot_sha256": "08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097",
    "window_start": "2026-01-01", "window_end": "2026-12-31", "mode": "validate",
}
for key, default in defaults.items():
    dbutils.widgets.text(key, default)
parameters = {key: dbutils.widgets.get(key).strip() for key in defaults}
root_parameter = parameters.pop("repo_root")
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser la ruta absoluta del repositorio completo en Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
root = next((p for p in roots if (p / "databricks/gtfs_gold/route_service_day.py").is_file()), None)
if root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root.")
module_path = str(root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from gtfs_gold.route_service_day import analysis_window, run_route_service_day
parameters["sha256"] = parameters.pop("snapshot_sha256")
analysis_window(parameters["window_start"], parameters["window_end"])
print("Programación por ruta y fecha; modo:", parameters["mode"])


In [ ]:
# BLOQUE 2 — Preparación auditada y publicación explícita
result = run_route_service_day(spark, **parameters)
print(json.dumps(result, ensure_ascii=False, indent=2))
if result["persisted"]:
    print("Publicación diaria comprobada. Revisa las consultas de consumo antes de importarla a Power BI.")
else:
    print("Validación completada; no escribí tablas ni auditorías Gold.")
